# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIRˆ² dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant) library, using reproducible and referenceable Croissant schema.

### Dataset Source

The dataset source is provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print("Dataset loaded successfully.")
print(f"Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")


## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We will print out all record set `@id`s and their available field and column `@id`s.

In [ ]:
# List all record sets and their fields using their '@id'
if not dataset.record_sets:
    print("No record sets found in the dataset metadata.\n")
else:
    for rs in dataset.record_sets:
        print("Record Set @id:", rs.id)
        print("  Name:", rs.name)
        print("  Description:", rs.description)
        print("  Fields/Columns:")
        for f in rs.fields:
            print(f"    - {f.id} ({f.name})")
        print()

Let's explicitly list the available record set `@id`s. If the list below is empty, ensure the schema URL is correct and public, or check the dataset docs for guided usage.

In [ ]:
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Available record set @id's:")
for rsid in record_set_ids:
    print(f"- {rsid}")

For demonstration, let's inspect a sample of records for one record set. Below, replace `<record_set_id>` with an available `@id` shown above.

In [ ]:
# Choose a record set to explore (change this to any valid @id printed above)
example_record_set_id = record_set_ids[0] if record_set_ids else None

if example_record_set_id:
    print(f"Showing first 2 records for record set: {example_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i >= 1:
            break
else:
    print("No record sets available to inspect records.")

## 3. Data Extraction

Extract data from all record sets into pandas DataFrames for subsequent analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}\nColumns: {df.columns.tolist()}")

# Show a preview of the first record set
if record_set_ids:
    df_preview = dataframes[record_set_ids[0]]
    print(f"\nSample records for record set {record_set_ids[0]}:")
    display(df_preview.head())
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)

Apply common processing steps, such as filtering, normalizing numeric fields, and grouping data for summarization. All field or column references use the `@id` from the schema.

In [ ]:
# --- Editable Parameters ---
# Choose the record set (by @id)
rs_id = example_record_set_id if example_record_set_id else (record_set_ids[0] if record_set_ids else None)
# Choose a numeric column by its @id (replace this if needed)
numeric_candidate_col = None

if rs_id:
    df = dataframes[rs_id]
    # Try to automatically detect a numeric column from the data
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_candidate_col = numeric_fields[0]
    else:
        print("No numeric fields found in the DataFrame for EDA.")

    if numeric_candidate_col:
        print(f"Performing EDA on numeric field: {numeric_candidate_col}\n")

        threshold = df[numeric_candidate_col].mean() if not df[numeric_candidate_col].isnull().all() else 0
        filtered_df = df[df[numeric_candidate_col] > threshold]
        print(f"Filtered records where {numeric_candidate_col} > {threshold:.2f} (mean): {len(filtered_df)} records\n")
        
        # Normalization
        normalized_col = f"{numeric_candidate_col}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_candidate_col] - filtered_df[numeric_candidate_col].mean()) / (
            filtered_df[numeric_candidate_col].std() if filtered_df[numeric_candidate_col].std() else 1)
        print(f"Normalized '{numeric_candidate_col}' field:\n", filtered_df[[numeric_candidate_col, normalized_col]].head())

        # Try grouping by a non-numeric field if present
        group_candidates = [col for col in df.columns if col != numeric_candidate_col and df[col].dtype == object]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_candidate_col].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_candidate_col}' by '{group_field}':\n", grouped_df.head())
    else:
        print("No suitable numeric field found for EDA")
else:
    print("No available record set for analysis.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. For example, histogram of the selected numeric field, bar plot for grouped means, etc.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show visualizations if EDA produced results
if rs_id and numeric_candidate_col:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of the numeric field
    axs[0].set_title(f"Distribution of {numeric_candidate_col}")
    filtered_df[numeric_candidate_col].plot.hist(ax=axs[0], bins=30)

    # If we had group_field, show means by group
    if 'grouped_df' in locals() and group_field:
        axs[1].set_title(f"Mean {numeric_candidate_col} by {group_field}")
        sns.barplot(x=group_field, y=numeric_candidate_col, data=grouped_df, ax=axs[1])
        axs[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, you successfully loaded and explored the FAIRˆ² dataset with `mlcroissant`, referencing all entities using their Croissant `@id` values for reproducibility. We listed record sets and fields, loaded records as DataFrames, conducted basic EDA including filtering and normalization, and visualized key statistics. For advanced analysis, reference further data documentation and adapt to additional record sets or columns as needed.